# 3W Dataset's General Presentation

This is a general presentation of the 3W Dataset, to the best of its authors' knowledge, the first realistic and public dataset with rare undesirable real events in oil wells that can be readily used as a benchmark dataset for development of machine learning techniques related to inherent difficulties of actual data.

For more information about the theory behind each version of the 3W Dataset, please refer to publications listed in [CITATION.md](../../../CITATION.md).

# 1. Introduction

This Jupyter Notebook presents the 3W Dataset 2.0.0 in a general way. For this, some tables, graphs, and statistics are presented.

# 2. Imports and Configurations

In [ ]:
import os
import sys
sys.path.append(os.path.join('..','..'))

import re
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from ThreeWToolkit.dataset import ParquetDataset, ParquetDatasetConfig

%matplotlib inline
%config InlineBackend.figure_format = 'svg'

dataset_path = "../../../dataset"

# 3. Instances' Structure

Below, all 3W dataset's instances are loaded and the first one of each knowledge source (real, simulated and hand-drawn) is partially displayed.

#### Real Instances:

In [ ]:
event_types = ["real"]
ds_config = ParquetDatasetConfig(path=dataset_path, event_type=event_types)
real_instances = ParquetDataset(ds_config)

print(f"\nFiltering events of types {event_types}, there are {len(real_instances)} events")

In [ ]:
real_instances[0].model_dump()

In [ ]:
real_instances[0].signal.head()

#### Simulated Instances:

In [ ]:
event_types = ["simulated"]
ds_config = ParquetDatasetConfig(path=dataset_path, event_type=event_types)
simulated_instances = ParquetDataset(ds_config)

print(f"\nFiltering events of types {event_types}, there are {len(simulated_instances)} events")

In [ ]:
simulated_instances[0].model_dump()

In [ ]:
simulated_instances[0].signal.head()

#### Drawn instances

In [ ]:
event_types = ["drawn"]
ds_config = ParquetDatasetConfig(path=dataset_path, event_type=event_types)
drawn_instances = ParquetDataset(ds_config)

print(f"\nFiltering events of types {event_types}, there are {len(drawn_instances)} events")

In [ ]:
drawn_instances[0].model_dump()

In [ ]:
drawn_instances[0].signal.head()

#### Explanation:

Each instance is stored in a [Parquet file](https://parquet.apache.org/docs/) and loaded into a pandas DataFrame as follows:

* All Parquet files are created and read with pandas functions, `pyarrow` engine and `brotli` compression;
* For each instance, timestamps corresponding to observations are stored in Parquet file as its index and loaded into pandas DataFrame as its index;
* Each observation is stored in a line of a Parquet file and loaded as a line of a pandas DataFrame; 
* All variables are stored as float in columns of Parquet files and loaded as float in columns of pandas DataFrame;
* All labels are stored as `Int64` (not `int64`) in columns of Parquet files and loaded as `Int64` (not `int64`) in columns of pandas DataFrame.

The variables and labels are as follows:

* **ABER-CKGL**: Opening of the GLCK (gas lift choke) [%];
* **ABER-CKP**: Opening of the PCK (production choke) [%];
* **ESTADO-DHSV**: State of the DHSV (downhole safety valve) [0, 0.5, or 1];
* **ESTADO-M1**: State of the PMV (production master valve) [0, 0.5, or 1];
* **ESTADO-M2**: State of the AMV (annulus master valve) [0, 0.5, or 1];
* **ESTADO-PXO**: State of the PXO (pig-crossover) valve [0, 0.5, or 1];
* **ESTADO-SDV-GL**: State of the gas lift SDV (shutdown valve) [0, 0.5, or 1];
* **ESTADO-SDV-P**: State of the production SDV (shutdown valve) [0, 0.5, or 1];
* **ESTADO-W1**: State of the PWV (production wing valve) [0, 0.5, or 1];
* **ESTADO-W2**: State of the AWV (annulus wing valve) [0, 0.5, or 1];
* **ESTADO-XO**: State of the XO (crossover) valve [0, 0.5, or 1];
* **P-ANULAR**: Pressure in the well annulus [Pa];
* **P-JUS-BS**: Downstream pressure of the SP (service pump) [Pa];
* **P-JUS-CKGL**: Downstream pressure of the GLCK (gas lift choke) [Pa];
* **P-JUS-CKP**: Downstream pressure of the PCK (production choke) [Pa];
* **P-MON-CKGL**: Upstream pressure of the GLCK (gas lift choke) [Pa];
* **P-MON-CKP**: Upstream pressure of the PCK (production choke) [Pa];
* **P-MON-SDV-P**: Upstream pressure of the production SDV (shutdown valve) [Pa];
* **P-PDG**: Pressure at the PDG (permanent downhole gauge) [Pa];
* **PT-P**: Downstream pressure of the PWV (production wing valve) in the production tube [Pa];
* **P-TPT**: Pressure at the TPT (temperature and pressure transducer) [Pa];
* **QBS**: Flow rate at the SP (service pump) [m3/s];
* **QGL**: Gas lift flow rate [m3/s];
* **T-JUS-CKP**: Downstream temperature of the PCK (production choke) [oC];
* **T-MON-CKP**: Upstream temperature of the PCK (production choke) [oC];
* **T-PDG**: Temperature at the PDG (permanent downhole gauge) [oC];
* **T-TPT**: Temperature at the TPT (temperature and pressure transducer) [oC];
* **class**: Label of the observation;
* **state**: Well operational status.

Other informations are also loaded into each pandas Dataframe:

* **label**: instance label (event type);
* **well**: well name. Hand-drawn and simulated instances have fixed names. Real instances have names masked with incremental id;
* **id**: instance identifier. Hand-drawn and simulated instances have incremental id. Each real instance has an id generated from its first timestamp.

More information about these variables can be obtained from the following publicly available documents:

* ***Option in Portuguese***: R.E.V. Vargas. Base de dados e benchmarks para prognóstico de anomalias em sistemas de elevação de petróleo. Universidade Federal do Espírito Santo. Doctoral thesis. 2019. https://github.com/petrobras/3W/raw/main/docs/doctoral_thesis_ricardo_vargas.pdf.
* ***Option in English***: B.G. Carvalho. Evaluating machine learning techniques for detection of flow instability events in offshore oil wells. Universidade Federal do Espírito Santo. Master's degree dissertation. 2021. https://github.com/petrobras/3W/raw/main/docs/master_degree_dissertation_bruno_carvalho.pdf.

# 4. Table of Instances

The following table shows the amount of instances that compose the 3W Dataset, by knowledge source (real, simulated and hand-drawn instances) and by instance label.

In [ ]:
LABEL_NAMES = {
    0: "Normal Operation",
    1: "Abrupt Increase of BSW",
    2: "Spurious Closure of DHSV",
    3: "Severe Slugging",
    4: "Flow Instability",
    5: "Rapid Productivity Loss",
    6: "Quick Restriction in PCK",
    7: "Scaling in PCK",
    8: "Hydrate in Production Line",
    9: "Hydrate in Service Line",
}

events = {"WELL":{}, "SIMULATED":{},"DRAWN":{}}

for event_type in ["real", "simulated", "drawn"]:
    config = ParquetDatasetConfig(path=dataset_path, event_type=[event_type])
    dataset = ParquetDataset(config)

    counter = {}

    for data in dataset:
        instance_id = int(data.metadata.get("file_name").parents[0].name)
        instance_name = LABEL_NAMES[instance_id]
        counter[instance_id] = counter.get(instance_id, 0) + 1

    events[event_type] = counter

    print(f"Event type: {event_type}")
    print(counter)
    print("-" * 100)

In [ ]:
def create_table_of_instances(real_counts, simulated_counts, drawn_counts):
    labels_sorted = sorted(LABEL_NAMES.keys())

    toi = pd.DataFrame(
        {
            "REAL": [real_counts.get(i, 0) for i in labels_sorted],
            "SIMULATED": [simulated_counts.get(i, 0) for i in labels_sorted],
            "HAND-DRAWN": [drawn_counts.get(i, 0) for i in labels_sorted],
        },
        index=[f"{i} - {LABEL_NAMES[i]}" for i in labels_sorted],
    )

    toi.index.name = "INSTANCE LABEL"
    toi.columns.name = "SOURCE"

    toi["TOTAL"] = toi.sum(axis=1)
    toi.loc["TOTAL"] = toi.sum(axis=0)

    return toi.astype(int)

In [ ]:
toi = create_table_of_instances(
    events["WELL"],
    events["SIMULATED"],
    events["DRAWN"]
)

toi

# 5. Rare Undesirable Events

Considering only **real instances** and **threshold of 1%**, the 3W Dataset has the following amount of instances with rare events.

In [ ]:
def filter_rare_undesirable_events(
    toi: pd.DataFrame,
    threshold: float = 0.01,
    simulated: bool = False,
    drawn: bool = False,
) -> pd.DataFrame:

    selected_sources = ["REAL"]
    if simulated:
        selected_sources.append("SIMULATED")
    if drawn:
        selected_sources.append("HAND-DRAWN")

    selected_total = toi.loc["TOTAL", selected_sources].sum()
    cutoff = threshold * selected_total

    base_df = toi.drop(index="TOTAL").copy()
    base_df = base_df[~base_df.index.astype(str).str.startswith("0 - ")].copy()

    selected_count_per_row = base_df[selected_sources].sum(axis=1)

    rare_mask = (selected_count_per_row > 0) & (selected_count_per_row < cutoff)
    rue = base_df.loc[rare_mask, ["REAL", "SIMULATED", "HAND-DRAWN", "TOTAL"]].copy()

    if rue.empty:
        rue = pd.DataFrame(
            [[0, 0, 0, 0]],
            index=["TOTAL"],
            columns=["REAL", "SIMULATED", "HAND-DRAWN", "TOTAL"],
        )
    else:
        rue.loc["TOTAL"] = rue.sum(axis=0)

    rue.index.name = "INSTANCE LABEL"
    rue.columns.name = "SOURCE"

    return rue.astype(int)

In [ ]:
threshold = 0.01

rue = filter_rare_undesirable_events(toi, threshold)
rue

If **simulated instances** are also considered, the amount of instances with rare events in 3W Dataset become the one listed below.

In [ ]:
# real + simulated
rue = filter_rare_undesirable_events(toi, threshold, simulated=True)
rue

After also considering the **hand-drawn instances**, we get the final amount of instances with rare events in 3W Dataset.

In [ ]:
# real + simulated + drawn
rue = filter_rare_undesirable_events(toi, threshold, simulated=True, drawn=True)
rue

# 6. Scatter Map of Real Instances

A scatter map with all the **real instances** is shown below. The oldest one occurred in the middle of 2011 and the most recent one in the middle of 2023. In addition to the total number of considered wells, this map provides an overview of the occurrences distributions of each instance over time and between wells.

In [ ]:
LABEL_NAMES = {
    0: "Normal Operation",
    1: "Abrupt Increase of BSW",
    2: "Spurious Closure of DHSV",
    3: "Severe Slugging",
    4: "Flow Instability",
    5: "Rapid Productivity Loss",
    6: "Quick Restriction in PCK",
    7: "Scaling in PCK",
    8: "Hydrate in Production Line",
    9: "Hydrate in Service Line",
}


def plot_real_instances_scatter_colored(real_instances):
    rows = []

    for instance in real_instances:
        file_name = instance.metadata.get("file_name", None)
        signal = instance.metadata.get("signal", None)

        well_id = None
        start_time = None
        label_id = None

        if file_name is not None:
            file_path = Path(file_name)
            stem = file_path.stem

            try:
                label_id = int(file_path.parents[0].name)
            except Exception:
                label_id = None

            well_match = re.search(r"(WELL-\d+)", stem)
            time_match = re.search(r"(\d{14})", stem)

            if well_match:
                well_id = well_match.group(1)

            if time_match:
                start_time = pd.to_datetime(time_match.group(1), format="%Y%m%d%H%M%S")

        if start_time is None and signal is not None and isinstance(signal, pd.DataFrame):
            if len(signal.index) > 0:
                start_time = pd.to_datetime(signal.index.min())

        if well_id is not None and start_time is not None:
            rows.append(
                {
                    "well_id": well_id,
                    "start_time": start_time,
                    "label_id": label_id,
                    "label_name": LABEL_NAMES.get(label_id, str(label_id)),
                }
            )

    df = pd.DataFrame(rows)

    if df.empty:
        raise ValueError("No valid real instances were found to plot.")

    df["well_num"] = df["well_id"].str.extract(r"(\d+)").astype(int)
    df = df.sort_values(["well_num", "start_time"]).reset_index(drop=True)

    wells_sorted = sorted(df["well_id"].unique(), key=lambda x: int(re.search(r"\d+", x).group()))
    well_to_y = {w: i for i, w in enumerate(wells_sorted)}
    df["y"] = df["well_id"].map(well_to_y)

    plt.figure(figsize=(15, 8))

    for label_name, group in df.groupby("label_name"):
        plt.scatter(group["start_time"], group["y"], s=20, alpha=0.8, label=label_name)

    plt.yticks(range(len(wells_sorted)), wells_sorted)
    plt.xlabel("Oldest timestamp of the instance")
    plt.ylabel("Well")
    plt.title("Scatter Map of Real Instances")
    plt.grid(True, alpha=0.25)
    plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

    return df

In [ ]:
scatter_df = plot_real_instances_scatter_colored(real_instances)

# 7. Some Statistics

The 3W dataset's fundamental aspects related to inherent difficulties of actual data are presented next.

In [ ]:
def _load_raw_instance_df(instance) -> pd.DataFrame:
    df = instance.signal

    if "timestamp" in df.columns:
        df = df.set_index("timestamp")

    return df


def _calc_single_instance_stats(instance) -> pd.Series:
    df = _load_raw_instance_df(instance)

    vars_cols = list(df.columns[:-1])

    frozen_cols = [col for col in vars_cols if col != "class"]

    n_vars = len(vars_cols)

    n_vars_missing = int(df[vars_cols].isnull().all(axis=0).sum())

    n_vars_frozen = 0
    for col in frozen_cols:
        s = df[col]
        u_values = s.unique()

        if len(u_values) == 1 and not pd.isna(u_values[0]):
            n_vars_frozen += 1

    n_obs = int(len(df))
    n_obs_unlabeled = int(df["class"].isnull().sum()) if "class" in df.columns else 0

    return pd.Series(
        {
            "N VARIABLES": n_vars,
            "MISSING VARIABLES": n_vars_missing,
            "FROZEN VARIABLES": n_vars_frozen,
            "N OBSERVATIONS": n_obs,
            "UNLABELED OBSERVATIONS": n_obs_unlabeled,
        },
        dtype="int64",
    )


def _aggregate_instance_stats(instances) -> pd.Series:
    base = pd.Series(
        {
            "N VARIABLES": 0,
            "MISSING VARIABLES": 0,
            "FROZEN VARIABLES": 0,
            "N OBSERVATIONS": 0,
            "UNLABELED OBSERVATIONS": 0,
        },
        dtype="int64",
    )

    if instances is None:
        return base

    stats_list = [
        _calc_single_instance_stats(instance)
        for instance in instances
    ]

    if not stats_list:
        return base

    return pd.DataFrame(stats_list).sum(numeric_only=True)


def calc_stats_instances(
    real_instances,
    simulated_instances=None,
    drawn_instances=None
) -> pd.DataFrame:
    simulated_instances = simulated_instances or []
    drawn_instances = drawn_instances or []

    total_stats = (
        _aggregate_instance_stats(real_instances)
        + _aggregate_instance_stats(simulated_instances)
        + _aggregate_instance_stats(drawn_instances)
    )

    total_variables = int(total_stats["N VARIABLES"])
    total_observations = int(total_stats["N OBSERVATIONS"])

    missing_variables = int(total_stats["MISSING VARIABLES"])
    frozen_variables = int(total_stats["FROZEN VARIABLES"])
    unlabeled_observations = int(total_stats["UNLABELED OBSERVATIONS"])

    def _format_percentage(amount: int, total: int) -> str:
        pct = 0.0 if total == 0 else 100.0 * amount / total
        return f"{pct:.2f}% of {total}"

    return pd.DataFrame(
        {
            "Amount": [
                missing_variables,
                frozen_variables,
                unlabeled_observations,
            ],
            "Percentage": [
                _format_percentage(missing_variables, total_variables),
                _format_percentage(frozen_variables, total_variables),
                _format_percentage(unlabeled_observations, total_observations),
            ],
        },
        index=[
            "Missing Variables",
            "Frozen Variables",
            "Unlabeled Observations",
        ],
    )

In [ ]:
stats = calc_stats_instances(
    real_instances,
    simulated_instances,
    drawn_instances
)
stats